In [1]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

20:24:45 [INFO] Device: cuda


In [2]:
CFG = {
    "task":        "lo",
    "dataset":     "kcnh2",
    "fp_type":     "ecfp4",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {
        "train": train_df,
        "test": test_df,
    }

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} "
        f"n_train_clusters={train_df['cluster'].nunique()} | "
        f"test={len(test_df)} "
        f"n_test_clusters={test_df['cluster'].nunique()}"
    )

20:24:45 [INFO] Loaded lo/kcnh2 fold 1: train=3313, test=406
20:24:45 [INFO] Fold 1 | train=3313 y_mean=5.662 y_std=0.561 n_train_clusters=1 | test=406 n_test_clusters=34
20:24:45 [INFO] Loaded lo/kcnh2 fold 2: train=3313, test=406
20:24:45 [INFO] Fold 2 | train=3313 y_mean=5.663 y_std=0.562 n_train_clusters=1 | test=406 n_test_clusters=34
20:24:45 [INFO] Loaded lo/kcnh2 fold 3: train=3313, test=406
20:24:45 [INFO] Fold 3 | train=3313 y_mean=5.662 y_std=0.561 n_train_clusters=1 | test=406 n_test_clusters=34


In [5]:
folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

20:24:45 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/ecfp4_train_1.npz
20:24:45 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/ecfp4_test_1.npz
20:24:45 [INFO] Fold 1 | X_train: (3313, 1024), X_test: (406, 1024) | scale_features=False | y_mean=5.662 y_std=0.561
20:24:45 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/ecfp4_train_2.npz
20:24:45 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/ecfp4_test_2.npz
20:24:45 [INFO] Fold 2 | X_train: (3313, 1024), X_test: (406, 1024) | scale_features=False | y_mean=5.663 y_std=0.562
20:24:45 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/ecfp4_train_3.npz
20:24:45 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/ecfp4_test_3.npz
20:24:45 [INFO] Fold 3 | X_train: (3313, 1024),

In [6]:
n_inner_trainings = N_ITER * CFG["inner_k"] * len(CFG["outer_folds"])
n_final_trainings = N_SEEDS * len(CFG["outer_folds"])

logger.info(
    f"Planned trainings: "
    f"{n_inner_trainings} inner-search runs + "
    f"{n_final_trainings} final retraining runs"
)

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(
    fold_results,
    title="MLP kcnh2 Lo — ecfp4",
)

20:24:45 [INFO] Planned trainings: 900 inner-search runs + 9 final retraining runs
20:24:45 [INFO] 
OUTER FOLD 1
20:24:48 [INFO]   [1/150] inner MSE=0.7928 (score=-0.7928) (3s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
20:24:51 [INFO]   [2/150] inner MSE=0.8219 (score=-0.8219) (3s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
20:24:53 [INFO]   [3/150] inner MSE=0.9130 (score=-0.9130) (1s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
20:24:55 [INFO]   [4/150] inner MSE=0.7971 (score=-0.7971) (2s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
20:24:56 [INFO]   [5/150] inner MSE=0.8671 (score=-0.8671) (1s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
20:25:06 [INFO]   [6/150] inner MSE=0.8124 (score=-0.8124) (10s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
20:25:11 [INFO]   [7/150] inner MSE=0.8152 (score=-0.8152) (5s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
20:25:17 [INFO]   [8/150] inner MSE=0.7924 (score=-0.7924) (5s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
20:25:22 [INFO]   [9/150] inner MSE=0.8342 (score=-0.8342) (5s) 


MLP kcnh2 Lo — ecfp4
Fold 1: mean_spearman=0.4036
Fold 2: mean_spearman=0.4165
Fold 3: mean_spearman=0.3709

Aggregated metrics:
  mean_spearman_mean: 0.397
  mean_spearman_std: 0.0192
  std_spearman_mean: 0.3651
  std_spearman_std: 0.0312
  mean_r2_mean: -0.8531
  mean_r2_std: 0.0883
  mean_mae_mean: 0.9114
  mean_mae_std: 0.0054
  n_clusters_mean: 34.0
  n_clusters_std: 0.0

Best hyperparameters per fold:
Fold 1: L=2 N=1024 r=1.0 act=leaky_relu dropout=0.5 bn=False init=kaiming lr=3e-03 wd=1e-03 bs=256
Fold 2: L=3 N=1024 r=0.5 act=relu dropout=0.0 bn=True init=kaiming lr=2e-03 wd=5e-03 bs=128
Fold 3: L=2 N=1024 r=0.5 act=relu dropout=0.5 bn=False init=xavier lr=2e-03 wd=0e+00 bs=64


{'mean_spearman_mean': np.float64(0.397),
 'mean_spearman_std': np.float64(0.0192),
 'std_spearman_mean': np.float64(0.3651),
 'std_spearman_std': np.float64(0.0312),
 'mean_r2_mean': np.float64(-0.8531),
 'mean_r2_std': np.float64(0.0883),
 'mean_mae_mean': np.float64(0.9114),
 'mean_mae_std': np.float64(0.0054),
 'n_clusters_mean': np.float64(34.0),
 'n_clusters_std': np.float64(0.0)}